# Sales Agent Demo

The Sales Agent is the third specialist, now registered alongside Finance and Sentiment in
the boss agent's roster (`backend/agents/boss/registry.py`).

## Its 8 tools

| Tool | Computes | Data reality |
|---|---|---|
| `query_revenue_by_period` | Revenue + order count per day/week/month, trend direction | **Excludes trailing partial periods from the trend calc** — see below |
| `calculate_aov` | Overall + monthly average order value | Computed directly |
| `sales_by_category` | Revenue and items sold per product category | Computed directly |
| `seller_sales_ranking` | Top sellers by revenue/order volume | Volume lens only — deliberately overlaps with Operations' future `seller_performance_score` (reliability lens), per CLAUDE.md |
| `conversion_funnel_analysis` | Order status breakdown, drop-off rate | **Not a true top-of-funnel conversion** — Olist has no browse/cart data, this proxies the *fulfillment* funnel (placed -> delivered vs. canceled/unavailable) |
| `repeat_purchase_rate` | Share of customers with 2+ orders | Uses `customer_unique_id` (person-level), not `customer_id` (Olist issues a new `customer_id` per order — using it here would silently read as ~0% repeat rate) |
| `seasonal_sales_pattern` | Revenue by calendar month, aggregated across years | Only 2016-2018 in the data — not enough years to cleanly separate seasonality from year-over-year growth |
| `cross_sell_opportunities` | Category pairs frequently bought together | Simple co-occurrence within an order, not a trained recommendation model |

## A real trend-analysis bug caught while building this

The naive "last 12 months" window for `query_revenue_by_period` initially reported revenue as
**declining**. Checking the raw monthly order counts explained why:

| Month | Orders |
|---|---|
| 2018-08 | 6,512 |
| 2018-09 | 16 |
| 2018-10 | 4 |

Olist's data collection simply stops mid-way through September 2018 — the last two months in
the raw window are a collection cutoff, not a demand collapse. `query_revenue_by_period` now
detects trailing periods whose order count is far below the window's median and excludes them
from the trend calculation (while still returning them in the raw data, flagged via
`excluded_partial_periods`), rather than reporting a false decline. This is exactly the kind of
thing a single-shot LLM summarizing raw numbers would likely miss — worth remembering as a
concrete case for why tool-backed, code-verified findings matter over pure LLM reasoning over
raw data.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.sales import SalesAgent
from backend.db import DatabricksClient

db = DatabricksClient()
agent = SalesAgent(db)

## Run the agent against live Delta tables

In [2]:
briefing = agent.run("How are sales trending?")

print(f"Agent: {briefing.agent.value}")
print(f"Execution time: {briefing.execution_time_ms:.0f}ms")
print(f"Tool calls: {len(briefing.tool_calls)} ({sum(1 for t in briefing.tool_calls if t.success)} succeeded)")

Agent: Sales
Execution time: 9504ms
Tool calls: 8 (8 succeeded)


## Inspect the findings

In [3]:
for f in briefing.findings:
    print(f"[{f.severity.upper()}] (confidence {f.confidence:.2f}) {f.claim}")
    print(f"  source: {f.source}\n")

[INFO] (confidence 0.85) Revenue is growing across the last 12 months
  source: query_revenue_by_period

[INFO] (confidence 0.90) Overall average order value is 137.42
  source: calculate_aov

[INFO] (confidence 0.90) Top category by revenue is health_beauty (1255695.13)
  source: sales_by_category

[INFO] (confidence 0.90) Top seller by revenue is 4869f7a5dfa277a7dca6462dcf3b52b2 (229237.63)
  source: seller_sales_ranking

[INFO] (confidence 0.65) 1.24% of orders drop off (canceled/unavailable) out of 99441 total
  source: conversion_funnel_analysis

[WARNING] (confidence 0.90) Repeat purchase rate is 3.04% (2888 of 94990 customers)
  source: repeat_purchase_rate

[INFO] (confidence 0.60) Peak sales month is May, trough is Sep
  source: seasonal_sales_pattern

[INFO] (confidence 0.70) Strongest cross-sell pair is bed_bath_table + furniture_decor (70 orders)
  source: cross_sell_opportunities



## Verify the trailing-period fix directly

`query_revenue_by_period`'s raw output should show the excluded partial months explicitly.

In [4]:
from backend.agents.sales import tools as sales_tools

revenue = sales_tools.query_revenue_by_period(db)
print(f"Trend: {revenue['trend_direction']}")
if "excluded_partial_periods" in revenue:
    print(f"Note: {revenue['note']}")
    for p in revenue["excluded_partial_periods"]:
        print(f"  excluded: {p['period']} - {p['order_count']} orders")

Trend: growing
Note: 1 trailing period(s) excluded from trend calc - order count far below the window median, consistent with a data-collection cutoff rather than a real drop
  excluded: 2018-09-01T00:00:00.000Z - 1 orders


## Through the boss agent

Three specialists now registered. A revenue-and-satisfaction query should pull in all three
that are relevant — Sales and Finance both speak to "the numbers," Sentiment speaks to
"happy."

In [5]:
from backend.agents.boss import BossAgent

boss = BossAgent(db)
rec = boss.run("Give me a full picture of business performance this year")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

if rec.dissents:
    print("\nDissents:")
    for d in rec.dissents:
        print(f"  - {[a.value for a in d.agents_involved]}: {d.summary}")

Agents invoked: ['Finance', 'Sales']
Overall confidence: 0.78

Synthesis:
Board Memo: FY23 Business Performance

1. Revenue & Growth
   - Finance's query_revenue_by_period tool reports revenue growth over the last 12 months, corroborated by Sales' query_revenue_by_period. 
   - Sales' calculate_aov tool indicates an average order value of $137.42, supporting healthy transaction size.

2. Order & Payment Health
   - Finance's payment_failure_rate tool shows a 1.24% order failure rate (canceled/unavailable) out of 99,441 orders, matching Sales' conversion_funnel_analysis drop‑off figure.
   - Finance's refund_impact_analysis indicates 1.68% of total payment value ($269,735.11) is at risk from canceled/unavailable orders.
   - Finance's detect_revenue_anomalies tool flagged 13 anomalous days out of 614, warranting investigation.

3. Cost & Margin
   - Finance's calculate_margin_trend tool reports improving contribution margin across the last 12 months.
   - Finance's calculate_cogs tool c

## What's pending

- Operations, Growth, Risk, Compliance/HR — 4 specialists remaining
- `seller_sales_ranking` (volume) vs. the future `seller_performance_score` (reliability) is the clearest planned demo of intentional cross-agent overlap once Operations exists